In [3]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error, mean_squared_error


### Import datasets

In [4]:
ratings_filtered = pd.read_csv("ratings_filtered.csv")
movies_filtered = pd.read_csv("movies_filtered.csv")

### Split into training and test

In [5]:
data_ratings_shuffled = ratings_filtered.sample(frac=1, random_state=42).reset_index(drop=True)

split_index = int(len(data_ratings_shuffled) * 0.8)
ratings_training = data_ratings_shuffled[:split_index]
ratings_evaluation = data_ratings_shuffled[split_index:]


ratings_training.to_csv("ratings_training.csv", index=False)
ratings_evaluation.to_csv("ratings_evaluation.csv", index=False)

print(f" Training size: {len(ratings_training)}")
print(f" Evaluation size: {len(ratings_evaluation)}")

 Training size: 9542856
 Evaluation size: 2385714


### Create the User/item matrix

In [7]:
user_item_matrix = ratings_training.pivot(index='userId', columns='movieId', values='rating').fillna(0)


In [8]:
user_item_matrix

movieId,1,2,3,4,5,6,7,8,9,10,...,292609,292611,292615,292617,292619,292625,292627,292731,292737,292755
userId,,,,,,,,,,,,,,,,,,,,,
10,2.5,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
28,4.0,3.0,4.0,0.0,2.0,3.0,3.0,3.0,0.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
35,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
65,2.0,0.0,1.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
70,4.0,3.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200877,5.0,3.0,2.0,0.0,0.0,5.0,0.0,0.0,0.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
200878,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
200906,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
user_item_matrix.isna().any().any()


np.False_

### Convert to sparse matrix 

In [12]:
user_item_sparse = csr_matrix(user_item_matrix.values, dtype=np.float32)

In [ ]:
print(f"Dense would use: {user_item_matrix.size * 8 / 1e9:.2f} GB")
print(f"Sparse uses: {user_item_sparse.data.nbytes / 1e6:.2f} MB")

Dense would use: 7.77 GB
Sparse uses: 38.17 MB


In [ ]:
user_item_matrix.to_csv("user_item_matrix.csv", index=True)

### Calculate Pearson correlation for user similarities and create user-similarity matrix

In [13]:
# Map user/movie IDs to row/column indices
user_to_index = {user: idx for idx, user in enumerate(user_item_matrix.index)}
index_to_user = {idx: user for user, idx in user_to_index.items()}
movie_to_index = {movie: idx for idx, movie in enumerate(user_item_matrix.columns)}
index_to_movie = {idx: movie for movie, idx in movie_to_index.items()}


In [14]:
# Center the ratings (subtract user means) - similar to Pearson correlation
def center_ratings(user_item_matrix):
    """Center ratings by subtracting user means"""
    user_means = user_item_matrix.mean(axis=1)
    centered_matrix = user_item_matrix.sub(user_means, axis=0)
    # Replace zeros with NaN for missing ratings (not rated)
    centered_matrix[user_item_matrix == 0] = np.nan
    return centered_matrix, user_means

# Calculate centered matrix and user means
centered_matrix, user_means = center_ratings(user_item_matrix)

# Convert to sparse matrix for efficient computation (fill NaN with 0 for sparse)
centered_sparse = csr_matrix(centered_matrix.fillna(0).values, dtype=np.float32)

# Calculate cosine similarity on centered ratings
user_similarity = cosine_similarity(centered_sparse)

# Convert similarity matrix to DataFrame for easier handling
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

In [15]:
user_similarity_df.head()

userId,10,28,35,65,70,109,128,151,158,187,...,200778,200785,200822,200849,200875,200877,200878,200906,200933,200943
userId,,,,,,,,,,,,,,,,,,,,,
10,1.000001,0.283438,0.211601,0.222131,0.267186,0.277222,0.215409,0.345153,0.146293,0.187249,...,0.331575,0.267984,0.232277,0.339328,0.160056,0.204631,0.096870,0.320634,0.292844,0.265394
28,0.283438,0.999993,0.233742,0.281587,0.308964,0.290728,0.233820,0.263289,0.291422,0.256741,...,0.315645,0.259704,0.265840,0.239975,0.258981,0.300324,0.201449,0.255059,0.326511,0.292205
35,0.211601,0.233742,0.999997,0.216739,0.181748,0.151965,0.119017,0.215034,0.128114,0.088808,...,0.246683,0.184559,0.193329,0.175774,0.190018,0.148524,0.123345,0.192232,0.254670,0.173806
65,0.222131,0.281587,0.216739,1.000001,0.278311,0.239655,0.201404,0.222980,0.196134,0.181080,...,0.298203,0.269405,0.247733,0.154721,0.229799,0.231393,0.179157,0.205160,0.276182,0.248941
70,0.267186,0.308964,0.181748,0.278311,1.000001,0.305881,0.211446,0.206940,0.257002,0.290080,...,0.269002,0.247316,0.268469,0.168064,0.185046,0.266623,0.170564,0.285936,0.265172,0.244777


In [16]:
user_similarity_df.to_csv("user_similarity.csv")

In [28]:
# Function to get top N similar users
def get_similar_users(user_id, n=10):
    """Get top N similar users for a given user"""
    if user_id not in user_to_index:
        return []
    
    user_idx = user_to_index[user_id]
    similar_users = user_similarity_df.iloc[user_idx].sort_values(ascending=False)
    # Exclude the user itself and users with zero similarity
    similar_users = similar_users.drop(user_id)
    similar_users = similar_users[similar_users > 0]
    
    return similar_users.head(n)


In [30]:
get_similar_users(10, n=5)

userId
191835    0.456165
87517     0.452921
158135    0.445491
177776    0.440944
86664     0.439468
Name: 10, dtype: float32

### Predicting ratings

In [31]:
# Function to predict rating using user-based collaborative filtering
def predict_rating(user_id, movie_id, k=10):
    """Predict rating for user-movie pair using k nearest neighbors"""
    if user_id not in user_to_index or movie_id not in movie_to_index:
        return user_means.get(user_id, 2.5)  # Default to average rating if user not found
    
    user_idx = user_to_index[user_id]
    movie_idx = movie_to_index[movie_id]
    
    # Get similar users
    similar_users = get_similar_users(user_id, k*2)  # Get more users to ensure we have enough raters
    
    if len(similar_users) == 0:
        return user_means[user_id]  # Return user mean if no similar users
    
    # Get users who have rated this movie
    movie_ratings = user_item_matrix.iloc[:, movie_idx]
    rated_users = movie_ratings[movie_ratings > 0].index
    
    # Filter similar users who have rated the movie
    valid_similar_users = similar_users.index.intersection(rated_users)
    
    if len(valid_similar_users) == 0:
        return user_means[user_id]  # Return user mean if no similar users rated the movie
    
    # Take top k similar users who rated the movie
    top_similar_users = similar_users.loc[valid_similar_users].head(k)
    
    if len(top_similar_users) == 0:
        return user_means[user_id]
    
    # Calculate weighted average of ratings
    numerator = 0
    denominator = 0
    
    for similar_user_id, similarity in top_similar_users.items():
        similar_user_idx = user_to_index[similar_user_id]
        rating = user_item_matrix.iloc[similar_user_idx, movie_idx]
        user_mean = user_means[similar_user_id]
        
        numerator += similarity * (rating - user_mean)
        denominator += abs(similarity)
    
    if denominator == 0:
        return user_means[user_id]
    
    predicted_rating = user_means[user_id] + (numerator / denominator)
    
    # Ensure rating is within valid range [0.5, 5]
    predicted_rating = max(0.5, min(5.0, predicted_rating))
    
    return predicted_rating


In [32]:
predict_rating(1, 31)  # Example prediction for userId=1 and movieId=31

2.5

### Evaluate the model 

In [52]:
def evaluate_and_store_predictions(test_data, k=10, sample_size=1000, random_state=42):
    """Evaluate the model on a sample of test data and store results in CSV"""
    # Sample a subset of the data for faster evaluation
    if len(test_data) > sample_size:
        test_sample = test_data.sample(sample_size, random_state=random_state)
        print(f"Sampling {sample_size} records from test data for faster evaluation")
    else:
        test_sample = test_data
    
    predictions = []
    actuals = []
    user_ids = []
    movie_ids = []
    
    print("Generating predictions...")
    for idx, row in test_sample.iterrows():
        user_id = row['userId']
        movie_id = row['movieId']
        actual_rating = row['rating']
        
        predicted_rating = predict_rating(user_id, movie_id, k)
        
        predictions.append(predicted_rating)
        actuals.append(actual_rating)
        user_ids.append(user_id)
        movie_ids.append(movie_id)
    
    # Calculate metrics
    mae = mean_absolute_error(actuals, predictions)
    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    
    # Create DataFrame with results
    results_df = pd.DataFrame({
        'userId': user_ids,
        'movieId': movie_ids,
        'actual_rating': actuals,
        'predicted_rating': predictions,
        'absolute_error': np.abs(np.array(actuals) - np.array(predictions)),
        'squared_error': (np.array(actuals) - np.array(predictions)) ** 2
    })
    
    # Save to CSV
    results_df.to_csv("collaborative_filtering_results.csv", index=False)
    print(f"Results saved to collaborative_filtering_results.csv")

    # Also save summary statistics
    summary_stats = {
        'MAE': mae,
        'RMSE': rmse,
        'total_predictions': len(predictions),
        'average_actual_rating': np.mean(actuals),
        'average_predicted_rating': np.mean(predictions),
        'sample_size': sample_size if len(test_data) > sample_size else len(test_data)
    }
    
    summary_df = pd.DataFrame([summary_stats])
    summary_df.to_csv("prediction_summary.csv", index=False)
    print("Summary statistics saved to prediction_summary.csv")
    
    return mae, rmse, results_df



In [58]:
# Evaluate on a subset
print("Evaluating model on subset and storing predictions...")
mae, rmse, results_df = evaluate_and_store_predictions(ratings_evaluation, k=100, sample_size=10000)
print(f"MAE: {mae:.4f}, RMSE: {rmse:.4f}")

Evaluating model on subset and storing predictions...
Sampling 10000 records from test data for faster evaluation
Generating predictions...
Results saved to collaborative_filtering_results.csv
Summary statistics saved to prediction_summary.csv
MAE: 0.7347, RMSE: 0.9974


In [64]:
def collaborative_filtering_recommendations(user_id, n=10, k=100):
    """Get top N movie recommendations for a user"""
    if user_id not in user_to_index:
        return []
    
    user_idx = user_to_index[user_id]
    
    # Get movies the user hasn't rated
    user_ratings = user_item_matrix.iloc[user_idx]
    unrated_movies = user_ratings[user_ratings == 0].index
    
    # Predict ratings for unrated movies
    predictions = []
    for movie_id in unrated_movies[:500]:  # Limit to top 500 for efficiency
        predicted_rating = predict_rating(user_id, movie_id, k)
        predictions.append((movie_id, predicted_rating))
    
    # Sort by predicted rating and return top N
    predictions.sort(key=lambda x: x[1], reverse=True)
    top_movies = [movie_id for movie_id, _ in predictions[:n]]
    
    # Join with metadata
    recommendations = movies_filtered[movies_filtered['movieId'].isin(top_movies)][
        ['movieId', 'title', 'genres']
    ]
    return recommendations

In [68]:
collaborative_filtering_recommendations(185551, n=5, k=100)  

,movieId,title,genres
79,80,"White Balloon, The (Badkonake sefid) (1995)",Children|Drama
211,213,Burnt by the Sun (Utomlyonnye solntsem) (1994),Drama
564,571,"Wedding Gift, The (1994)",Drama|Romance
600,608,Fargo (1996),Comedy|Crime|Drama|Thriller
607,615,Bread and Chocolate (Pane e cioccolata) (1973),Comedy|Drama
